In [ ]:
# Đọc file, kiểm tra nhanh: đủ 3 nhãn chưa, có null không
import pandas as pd
df = pd.read_csv('reviews_labeled.csv')
print(df['label'].value_counts())
print(df['clean_text'].isnull().sum())

# Chia dữ liệu thành 3 tập theo tỉ lệ 70/15/15:
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label']

# Tách 70% train, 30% còn lại
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Tách 30% còn lại thành val 15% và test 15%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# Khởi tạo vectorizer unigram và fit trên tập train:
from sklearn.feature_extraction.text import TfidfVectorizer

# Unigram: mỗi token là 1 từ đơn
tfidf_unigram = TfidfVectorizer(
    ngram_range=(1, 1),     # chỉ unigram
    max_features=50000,     # giữ 50k từ phổ biến nhất (tránh quá nặng)
    min_df=2                # bỏ từ xuất hiện < 2 lần
)

X_train_uni = tfidf_unigram.fit_transform(X_train)  # fit + transform train
X_val_uni   = tfidf_unigram.transform(X_val)         # chỉ transform, không fit lại
X_test_uni  = tfidf_unigram.transform(X_test)

print(f"Kích thước ma trận unigram train: {X_train_uni.shape}")

In [ ]:
# Khởi tạo vectorizer bigram tương tự:
# Bigram: mỗi token là cặp 2 từ liên tiếp
tfidf_bigram = TfidfVectorizer(
    ngram_range=(1, 2),     # unigram + bigram
    max_features=50000,
    min_df=2
)

X_train_bi = tfidf_bigram.fit_transform(X_train)
X_val_bi   = tfidf_bigram.transform(X_val)
X_test_bi  = tfidf_bigram.transform(X_test)

print(f"Kích thước ma trận bigram train: {X_train_bi.shape}")

In [ ]:
# Lưu 2 vectorizer vào thư mục models/ để dùng lại:
import joblib
import os

os.makedirs('models', exist_ok=True)
joblib.dump(tfidf_unigram, 'models/tfidf_unigram.pkl')
joblib.dump(tfidf_bigram,  'models/tfidf_bigram.pkl')

# Lưu labels và split data vào file để dùng chung:
import numpy as np
from scipy.sparse import save_npz

save_npz('models/X_train_uni.npz', X_train_uni)
save_npz('models/X_val_uni.npz',   X_val_uni)
save_npz('models/X_test_uni.npz',  X_test_uni)

save_npz('models/X_train_bi.npz',  X_train_bi)
save_npz('models/X_val_bi.npz',    X_val_bi)
save_npz('models/X_test_bi.npz',   X_test_bi)

# Lưu labels
y_train.to_csv('models/y_train.csv', index=False)
y_val.to_csv('models/y_val.csv',     index=False)
y_test.to_csv('models/y_test.csv',   index=False)

In [ ]:
import joblib
import pandas as pd
from scipy.sparse import load_npz

X_train_uni = load_npz('models/X_train_uni.npz')
X_val_uni   = load_npz('models/X_val_uni.npz')
X_train_bi  = load_npz('models/X_train_bi.npz')
X_val_bi    = load_npz('models/X_val_bi.npz')
y_train = pd.read_csv('models/y_train.csv').squeeze()
y_val   = pd.read_csv('models/y_val.csv').squeeze()

In [ ]:
# Thử nghiệm 3 giá trị alpha, ghi lại Macro F1 mỗi lần:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score

results = []

for alpha in [0.1, 0.5, 1.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_uni, y_train)
    y_pred = model.predict(X_val_uni)
    f1 = f1_score(y_val, y_pred, average='macro')
    results.append({'model': 'NB', 'ngram': 'unigram', 'alpha': alpha, 'macro_f1': round(f1, 4)})
    print(f"alpha={alpha} | Macro F1 (unigram): {f1:.4f}")
# Chọn alpha tốt nhất, train lại và lưu mô hình:
best_alpha_uni = 0.5    # thay bằng alpha cho F1 cao nhất

nb_unigram = MultinomialNB(alpha=best_alpha_uni)
nb_unigram.fit(X_train_uni, y_train)
joblib.dump(nb_unigram, 'models/nb_unigram.pkl')

In [ ]:
# Lặp lại tương tự với bigram:
for alpha in [0.1, 0.5, 1.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_bi, y_train)
    y_pred = model.predict(X_val_bi)
    f1 = f1_score(y_val, y_pred, average='macro')
    results.append({'model': 'NB', 'ngram': 'bigram', 'alpha': alpha, 'macro_f1': round(f1, 4)})
    print(f"alpha={alpha} | Macro F1 (bigram): {f1:.4f}")

best_alpha_bi = 0.5     # thay bằng alpha cho F1 cao nhất

nb_bigram = MultinomialNB(alpha=best_alpha_bi)
nb_bigram.fit(X_train_bi, y_train)
joblib.dump(nb_bigram, 'models/nb_bigram.pkl')

In [ ]:
import pandas as pd
df_results = pd.DataFrame(results)
df_results.to_csv('models/nb_results.csv', index=False)
print(df_results)

In [ ]:
import joblib
import pandas as pd
from scipy.sparse import load_npz
X_train_uni = load_npz('models/X_train_uni.npz')
X_val_uni   = load_npz('models/X_val_uni.npz')
X_train_bi  = load_npz('models/X_train_bi.npz')
X_val_bi    = load_npz('models/X_val_bi.npz')
y_train = pd.read_csv('models/y_train.csv').squeeze()
y_val   = pd.read_csv('models/y_val.csv').squeeze()

In [ ]:
# Thử nghiệm 3 giá trị C, ghi lại Macro F1 mỗi lần:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
results = []
for C in [0.1, 1.0, 10.0]:
    model = LogisticRegression(
        C=C,
        max_iter=1000,          # tăng lên để đảm bảo hội tụ
        random_state=42,
        class_weight='balanced' # xử lý mất cân bằng lớp
    )
    model.fit(X_train_uni, y_train)
    y_pred = model.predict(X_val_uni)
    f1 = f1_score(y_val, y_pred, average='macro')
    results.append({'model': 'LR', 'ngram': 'unigram', 'C': C, 'macro_f1': round(f1, 4)})
    print(f"C={C} | Macro F1 (unigram): {f1:.4f}")
# Chọn C tốt nhất, train lại và lưu mô hình:
best_C_uni = 1.0    # thay bằng C cho F1 cao nhất
lr_unigram = LogisticRegression(C=best_C_uni, max_iter=1000,
                                 random_state=42, class_weight='balanced')
lr_unigram.fit(X_train_uni, y_train)
joblib.dump(lr_unigram, 'models/lr_unigram.pkl')

In [ ]:
# Lặp lại tương tự với bigram:
for C in [0.1, 1.0, 10.0]:
    model = LogisticRegression(C=C, max_iter=1000,
                                random_state=42, class_weight='balanced')
    model.fit(X_train_bi, y_train)
    y_pred = model.predict(X_val_bi)
    f1 = f1_score(y_val, y_pred, average='macro')
    results.append({'model': 'LR', 'ngram': 'bigram', 'C': C, 'macro_f1': round(f1, 4)})
    print(f"C={C} | Macro F1 (bigram): {f1:.4f}")
best_C_bi = 1.0     # thay bằng C cho F1 cao nhất
lr_bigram = LogisticRegression(C=best_C_bi, max_iter=1000,
                                random_state=42, class_weight='balanced')
lr_bigram.fit(X_train_bi, y_train)
joblib.dump(lr_bigram, 'models/lr_bigram.pkl')

In [ ]:
df_results = pd.DataFrame(results)
df_results.to_csv('models/lr_results.csv', index=False)
print(df_results)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df_nb = pd.read_csv('models/nb_results.csv')
df_lr = pd.read_csv('models/lr_results.csv')
df_all = pd.concat([df_nb, df_lr], ignore_index=True)
print(df_all)

In [ ]:
# Lọc ra kết quả tốt nhất của mỗi tổ hợp (mô hình × n-gram):
# Lấy kết quả tốt nhất (alpha/C tốt nhất) của từng tổ hợp
best = df_all.groupby(['model', 'ngram'])['macro_f1'].max().reset_index()
best = best.sort_values('macro_f1', ascending=False)
print("\n===== BẢNG SO SÁNH 4 TỔ HỢP =====")
print(best.to_string(index=False))

In [ ]:
# Vẽ biểu đồ cột nhóm (grouped bar chart) để nhìn trực quan:
fig, ax = plt.subplots(figsize=(8, 5))
models = ['NB Unigram', 'NB Bigram', 'LR Unigram', 'LR Bigram']
f1_scores = [
    best[(best['model']=='NB') & (best['ngram']=='unigram')]['macro_f1'].values[0],
    best[(best['model']=='NB') & (best['ngram']=='bigram')]['macro_f1'].values[0],
    best[(best['model']=='LR') & (best['ngram']=='unigram')]['macro_f1'].values[0],
    best[(best['model']=='LR') & (best['ngram']=='bigram')]['macro_f1'].values[0],
]
colors = ['#5DCAA5', '#1D9E75', '#85B7EB', '#378ADD']
bars = ax.bar(models, f1_scores, color=colors, width=0.5)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Macro F1-score')
ax.set_title('So sánh Macro F1 — 4 tổ hợp mô hình')
for bar, score in zip(bars, f1_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('output/bieu_do_so_sanh.png', dpi=150)
plt.show()

Mô hình Logistic Regression (LR) cho chỉ số Macro F1-score vượt trội hoàn toàn so với Naive Bayes (NB) ở cả hai cấu hình n-gram.
Giải thích: 
- Logistic Regression (LR) là mô hình phân loại discriminative (phân biệt). Nó tối ưu hóa trực tiếp ranh giới quyết định và học các trọng số độc lập cho từng từ dựa trên mối tương quan tổng thể mà không bị ràng buộc bởi giả định phi thực tế.
- Trong khi đó, Naive Bayes (NB) dựa trên giả định các đặc trưng (từ vựng) độc lập có điều kiện với nhau. Trong văn bản thực tế, các từ luôn có sự liên kết chặt chẽ về mặt ngữ nghĩa và ngữ pháp. Giả định "ngây thơ" này của NB khiến mô hình bị mất đi các thông tin ngữ cảnh quan trọng, dẫn đến kết quả phân loại kém hơn rõ rệt.

Cấu hình Bigram mang lại hiệu quả tốt hơn hẳn so với Unigram trên cả hai mô hình
Đánh giá sự cải thiện: 
- Bigram thực sự cải thiện kết quả. Việc bổ sung các cụm 2 từ đi liền nhau giúp mô hình giữ lại được các liên kết ngữ nghĩa quan trọng (ví dụ: các cụm phủ định như "không thích", hoặc nhấn mạnh như "rất tốt").
- Đối với bài toán phân tích sắc thái (Sentiment Analysis), những cụm từ ghép này mang tính quyết định để phân biệt cảm xúc, điều mà Unigram (xét riêng lẻ từng từ "không" và "thích") thường bỏ sót hoặc nhận diện sai lệch.

Thông thường đối với các tập dữ liệu đánh giá/sắc thái bằng tiếng Việt, lớp Neutral (Trung lập) là lớp khó phân loại nhất.
Giải thích:
- Sự mơ hồ về mặt ngữ nghĩa: Các văn bản thuộc lớp Neutral thường chứa lẫn lộn cả từ ngữ tích cực lẫn tiêu cực (ví dụ: "máy dùng ổn nhưng giao hàng hơi chậm"), hoặc hoàn toàn là các câu hỏi/câu mô tả thông số khách quan không mang cảm xúc rõ rệt. Điều này khiến mô hình rất dễ dự đoán nhầm lớp Neutral sang Positive hoặc Negative.
- Mất cân bằng dữ liệu: Lớp Neutral thường có số lượng mẫu (sample size) ít hơn nhiều so với hai lớp cực đoan là Positive và Negative, khiến mô hình không được huấn luyện đủ để nhận diện tốt ranh giới của lớp này.